# LSTM for Markowitz Portfolio

### Objectives
- Understand how different estimators of parameters (mu, Sigma) change Markowitz optimal weights.
- Compare a historical baseline estimator vs a simple LSTM estimator.
- Compare both with an ex-post oracle benchmark.

### What we will build
- Case A (Baseline): mu and Sigma from historical sample mean/cov in (t-n ... t-1).
- Case B (DL): mu from LSTM next-day prediction (per asset), Sigma from historical covariance (simple default).
- Oracle case: mu_true and Sigma_true from future window (t ... t+m), only as benchmark.
- Three Markowitz portfolios (same criterion), then comparison metrics.

### Important notice
- This notebook is educational. It is **not** financial advice.
- The oracle uses future realized data, so it is **not investable**.


# 1. Main idea (simple ASCII drawing)

```
Past window                Decision day          Future window (oracle)
(t-n ... t-1)                 t0                 (t0 ... t0+m)
     |-------------------------|-------------------------|
     estimate mu, Sigma        build portfolio           oracle benchmark

Case A: mu_A, Sigma_A from past sample mean/cov
Case B: mu_B from LSTM (one-step prediction at t0), Sigma_B = Sigma_A
Oracle: mu_true, Sigma_true from future realized returns

Then compare w_A and w_B versus w_T (oracle optimal weights).
```


In [1]:
# 2. Imports + basic configuration

import importlib.util

# Install yfinance in Colab if needed
if importlib.util.find_spec("yfinance") is None:
    !pip -q install yfinance

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import yfinance as yf
from scipy.optimize import minimize
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

print("Libraries ready.")


Libraries ready.


# 3. Step 1: Download ETF data and prepare returns

We will use adjusted close prices (Adj Close).

Why Adj Close?
- It adjusts for splits/dividends.
- It is usually better for return calculations.

Then we compute daily returns:
- `return_t = price_t / price_{t-1} - 1`


In [2]:
# 4. Download Adj Close prices and compute daily returns

tickers = ["SPY", "QQQ", "TLT", "GLD"]
start_date = "2000-01-01"
end_date = None  # up to today

prices = yf.download(tickers, start=start_date, end=end_date, auto_adjust=False, progress=False)["Adj Close"].copy()
prices = prices.dropna(how="any")

returns = prices.pct_change().dropna().copy()

print("Prices shape:", prices.shape)
print("Returns shape:", returns.shape)
display(prices.head())
display(returns.head())


Prices shape: (5385, 4)
Returns shape: (5384, 4)


Ticker,GLD,QQQ,SPY,TLT
Date,,,,
2004-11-18,44.380001,33.156471,79.950729,44.324734
2004-11-19,44.779999,32.641716,79.061920,43.971016
2004-11-22,44.950001,32.953949,79.438980,44.200195
2004-11-23,44.750000,32.903297,79.560181,44.254997
2004-11-24,45.049999,33.190235,79.748711,44.254997


Ticker,GLD,QQQ,SPY,TLT
Date,,,,
2004-11-19,0.009013,-0.015525,-0.011117,-0.007980
2004-11-22,0.003796,0.009565,0.004769,0.005212
2004-11-23,-0.004449,-0.001537,0.001526,0.001240
2004-11-24,0.006704,0.008721,0.002370,0.000000
2004-11-26,0.005327,-0.003052,-0.000760,-0.006529


In [3]:
# 5. Exercise 1 (TODO)
# Add one extra ticker (for example, EEM) and recompute returns.

tickers = ["SPY", "QQQ", "TLT", "GLD", "EEM"]

prices = yf.download(tickers, start=start_date, end=end_date, auto_adjust=False, progress=False)["Adj Close"].copy()
prices = prices.dropna(how="any")

returns = prices.pct_change().dropna().copy()

print("Prices shape:", prices.shape)
print("Returns shape:", returns.shape)

display(prices.head())
display(returns.head())

Prices shape: (5385, 5)
Returns shape: (5384, 5)


Ticker,EEM,GLD,QQQ,SPY,TLT
Date,,,,,
2004-11-18,13.853825,44.380001,33.156475,79.950714,44.324772
2004-11-19,13.655754,44.779999,32.641720,79.061928,43.971043
2004-11-22,13.709060,44.950001,32.953930,79.438995,44.200184
2004-11-23,13.732822,44.750000,32.903309,79.560181,44.255016
2004-11-24,13.879029,45.049999,33.190235,79.748726,44.255016


Ticker,EEM,GLD,QQQ,SPY,TLT
Date,,,,,
2004-11-19,-0.014297,0.009013,-0.015525,-0.011117,-0.007980
2004-11-22,0.003904,0.003796,0.009565,0.004769,0.005211
2004-11-23,0.001733,-0.004449,-0.001536,0.001526,0.001241
2004-11-24,0.010647,0.006704,0.008720,0.002370,0.000000
2004-11-26,0.016607,0.005327,-0.003051,-0.000760,-0.006530


# 6. Step 2: Choose t0, n and m

Definitions:
- `t0`: date when we decide the portfolio.
- `n`: lookback window length (past days).
- `m`: oracle future window length (future days).

Past window for estimators: `(t0-n ... t0-1)`
Future oracle window: `(t0 ... t0+m)`


In [4]:
# 7. Define n, m and select one evaluation date t0

n = 252   # about 1 trading year
m = 60    # about 3 months

if len(returns) <= n + m + 1:
    raise ValueError("Not enough data for chosen n and m.")

# Choose t0 near the end, but leave m future days available
idx_t0 = len(returns) - m - 1
t0 = returns.index[idx_t0]

hist_window = returns.iloc[idx_t0 - n: idx_t0].copy()
future_window = returns.iloc[idx_t0: idx_t0 + m].copy()

print("Chosen t0:", t0.date())
print("Historical window:", hist_window.index.min().date(), "to", hist_window.index.max().date(), "| rows =", len(hist_window))
print("Future oracle window:", future_window.index.min().date(), "to", future_window.index.max().date(), "| rows =", len(future_window))


Chosen t0: 2026-01-20
Historical window: 2025-01-16 to 2026-01-16 | rows = 252
Future oracle window: 2026-01-20 to 2026-04-15 | rows = 60


In [5]:
# 8. Exercise 2 (TODO)
# Change n to 126 and see how mu_A changes.

n = 126

idx_t0 = len(returns) - m - 1
hist_window = returns.iloc[idx_t0 - n: idx_t0].copy()

mu_A = hist_window.mean()

print("mu_A with n=126:")
display(mu_A.to_frame("mu_A"))

mu_A with n=126:


,mu_A
Ticker,
EEM,0.001459
GLD,0.002561
QQQ,0.000870
SPY,0.000840
TLT,0.000427


# 9. Step 3: Case A (baseline) estimate mu and Sigma from history

Simple idea:
- `mu_A`: average daily return in the past window.
- `Sigma_A`: covariance matrix of daily returns in the past window.

These are sample estimates from historical data.


In [6]:
# 10. Compute mu_A and Sigma_A from historical window

mu_A = hist_window.mean()
Sigma_A = hist_window.cov()

print("mu_A (daily expected returns):")
display(mu_A.to_frame(name="mu_A"))

print("Sigma_A (daily covariance matrix):")
display(Sigma_A)


mu_A (daily expected returns):


,mu_A
Ticker,
EEM,0.001459
GLD,0.002561
QQQ,0.000870
SPY,0.000840
TLT,0.000427


Sigma_A (daily covariance matrix):


Ticker,EEM,GLD,QQQ,SPY,TLT
Ticker,,,,,
EEM,0.000076,0.000039,0.000058,0.000043,0.000000
GLD,0.000039,0.000164,0.000013,0.000011,0.000006
QQQ,0.000058,0.000013,0.000088,0.000061,-0.000003
SPY,0.000043,0.000011,0.000061,0.000046,-0.000002
TLT,0.000000,0.000006,-0.000003,-0.000002,0.000034


# 11. Step 4: Case B (LSTM) to estimate mu_B and Sigma_B

Now we make the LSTM more flexible and still simple for beginners.

Main idea:
- We choose the feature set with a simple string (for example, `feature_mode = "returns_vol_momentum"`).
- The notebook builds the input table automatically from that choice.
- Then it trains one small LSTM per asset for mean (`mu_B`) and one for risk proxy (next-day squared return).

Feature modes available:
- `"returns_only"`: only daily return of the asset.
- `"returns_vol"`: returns + rolling volatility features.
- `"returns_vol_momentum"`: returns + volatility + momentum + market return feature.

For covariance in Case B:
- Diagonal variances come from LSTM predicted risk per asset.
- Off-diagonal dependence uses historical correlations from the lookback window.


In [7]:
# 12. Helper functions for configurable LSTM features and sequence building

def build_asset_features(hist_returns, asset, feature_mode="returns_only"):
    """
    Build a feature DataFrame for one asset using only historical window data.

    Parameters
    ----------
    hist_returns : pd.DataFrame
        Historical daily returns in (t0-n ... t0-1).
    asset : str
        Asset ticker (for example: 'SPY').
    feature_mode : str
        One of: 'returns_only', 'returns_vol', 'returns_vol_momentum'.

    Returns
    -------
    pd.DataFrame
        Feature table indexed by date.
    """
    r = hist_returns[asset]

    feats = pd.DataFrame(index=hist_returns.index)

    # Always include the asset daily return itself
    feats["ret_1"] = r

    if feature_mode in ["returns_vol", "returns_vol_momentum"]:
        feats["ret_5"] = r.rolling(5).mean()
        feats["vol_10"] = r.rolling(10).std()

    if feature_mode == "returns_vol_momentum":
        feats["ret_20"] = r.rolling(20).mean()
        feats["vol_20"] = r.rolling(20).std()
        feats["mom_10"] = r.rolling(10).sum()

        # Simple market feature: SPY daily return
        # If asset is SPY, this is still valid (it becomes a self-market proxy).
        feats["mkt_ret_1"] = hist_returns["SPY"]

    if feature_mode not in ["returns_only", "returns_vol", "returns_vol_momentum"]:
        raise ValueError("Unknown feature_mode. Use: returns_only, returns_vol, returns_vol_momentum")

    return feats


def make_lstm_sequences(X_2d, y_1d, lookback):
    """
    Convert tabular data into LSTM sequences.

    X_2d shape: (T, n_features)
    y_1d shape: (T,)

    Output:
    - X_seq shape: (samples, lookback, n_features)
    - y_seq shape: (samples,)
    """
    X_seq, y_seq = [], []
    for i in range(lookback, len(X_2d)):
        X_seq.append(X_2d[i - lookback:i, :])
        y_seq.append(y_1d[i])

    return np.array(X_seq), np.array(y_seq)


def train_predict_one_step(hist_returns, asset, feature_mode, target_kind, lookback=20, max_epochs=20, batch_size=16):
    """
    Train one small LSTM using only historical window and predict value at t0.

    target_kind:
    - 'return'    -> target is next-day return r_{t+1}
    - 'sq_return' -> target is next-day squared return (r_{t+1})^2
    """
    # 1) Build features from historical window
    feat_raw = build_asset_features(hist_returns, asset, feature_mode=feature_mode)

    # 2) Build target for next day inside historical window
    if target_kind == "return":
        y_raw = hist_returns[asset].shift(-1)
    elif target_kind == "sq_return":
        y_raw = (hist_returns[asset] ** 2).shift(-1)
    else:
        raise ValueError("target_kind must be 'return' or 'sq_return'")

    # 3) Align and remove NaN rows (from rolling features and shift)
    data = feat_raw.copy()
    data["target"] = y_raw
    data = data.dropna().copy()

    if len(data) <= lookback + 5:
        raise ValueError(f"Not enough rows for LSTM in {asset} with mode={feature_mode}.")

    X_train_raw = data.drop(columns=["target"]).values
    y_train_raw = data["target"].values.reshape(-1, 1)

    # 4) Scale features and target separately (fit only on train window)
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X_train_scaled = x_scaler.fit_transform(X_train_raw)
    y_train_scaled = y_scaler.fit_transform(y_train_raw).flatten()

    # 5) Build LSTM sequences
    X_seq, y_seq = make_lstm_sequences(X_train_scaled, y_train_scaled, lookback)

    # 6) Small model (fast on CPU)
    model = Sequential([
        Input(shape=(lookback, X_seq.shape[2])),
        LSTM(32, return_sequences=True, dropout=0.1),
        LSTM(16, return_sequences=True, dropout=0.1),
        Dense(8, activation="relu"),
        Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse")

    early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

    model.fit(
        X_seq, y_seq,
        epochs=max_epochs,
        batch_size=batch_size,
        validation_split=0.1,
        callbacks=[early_stop],
        verbose=0
    )

    # 7) Build one-step input at t0 from the last 'lookback' historical rows
    feat_for_pred = feat_raw.dropna().copy()
    X_pred_scaled_all = x_scaler.transform(feat_for_pred.values)
    x_last = X_pred_scaled_all[-lookback:].reshape(1, lookback, X_seq.shape[2])

    y_pred_scaled = model.predict(x_last, verbose=0).flatten()[0]
    y_pred = y_scaler.inverse_transform(np.array([[y_pred_scaled]])).flatten()[0]

    return float(y_pred)


In [8]:
# 13. Configure feature modes, train LSTMs, and build mu_B and Sigma_B

lookback_LSTM = 40
max_epochs = 60
batch_size = 16

# -----------------------------
# Student-friendly configuration
# Change these strings to test different feature sets.
# -----------------------------
feature_mode_return = "returns_vol_momentum"
feature_mode_risk = "returns_vol_momentum"

print("Selected feature mode for return model:", feature_mode_return)
print("Selected feature mode for risk model:", feature_mode_risk)

mu_B_dict = {}
var_B_dict = {}

# Keep historical correlation structure (simple and stable)
corr_A = hist_window.corr().reindex(index=tickers, columns=tickers)

for asset in tickers:
    pred_ret = train_predict_one_step(
        hist_returns=hist_window,
        asset=asset,
        feature_mode=feature_mode_return,
        target_kind="return",
        lookback=lookback_LSTM,
        max_epochs=max_epochs,
        batch_size=batch_size
    )

    pred_sqret = train_predict_one_step(
        hist_returns=hist_window,
        asset=asset,
        feature_mode=feature_mode_risk,
        target_kind="sq_return",
        lookback=lookback_LSTM,
        max_epochs=max_epochs,
        batch_size=batch_size
    )

    mu_B_dict[asset] = pred_ret
    var_B_dict[asset] = max(pred_sqret, 1e-8)  # keep non-negative variance

mu_B = pd.Series(mu_B_dict).reindex(tickers)
var_B = pd.Series(var_B_dict).reindex(tickers)

# Build Sigma_B with predicted std devs and historical correlations
std_B = np.sqrt(var_B.values)
D_B = np.diag(std_B)
Sigma_B_values = D_B @ corr_A.values @ D_B
Sigma_B = pd.DataFrame(Sigma_B_values, index=tickers, columns=tickers)
Sigma_B = (Sigma_B + Sigma_B.T) / 2

print("mu_B (LSTM predicted next-day returns at t0):")
display(mu_B.to_frame(name="mu_B"))

print("var_B (LSTM predicted next-day squared returns at t0):")
display(var_B.to_frame(name="var_B"))

print("Sigma_B (from predicted variances + historical correlations):")
display(Sigma_B)


Selected feature mode for return model: returns_vol_momentum
Selected feature mode for risk model: returns_vol_momentum


mu_B (LSTM predicted next-day returns at t0):


,mu_B
SPY,0.000695
QQQ,0.000639
TLT,0.000230
GLD,0.003080
EEM,0.001634


var_B (LSTM predicted next-day squared returns at t0):


,var_B
SPY,0.000047
QQQ,0.000093
TLT,0.000027
GLD,0.000186
EEM,0.000080


Sigma_B (from predicted variances + historical correlations):


,SPY,QQQ,TLT,GLD,EEM
SPY,0.000047,0.000062,-0.000002,0.000011,0.000045
QQQ,0.000062,0.000093,-0.000003,0.000014,0.000061
TLT,-0.000002,-0.000003,0.000027,0.000006,0.000000
GLD,0.000011,0.000014,0.000006,0.000186,0.000043
EEM,0.000045,0.000061,0.000000,0.000043,0.000080


In [ ]:
# 14. Exercise 3 (TODO)
# Change the feature mode or lookback and retrain for one asset.
# Questions:
# 1) If you use fewer features, does prediction change?
# 2) If you change lookback from 20 to 10, what happens?

asset = "SPY"
results = []

for feature_mode in ["returns_only", "returns_vol", "returns_vol_momentum"]:
    for lookback in [10, 20, 40]:
        pred = train_predict_one_step(
            hist_returns=hist_window,
            asset=asset,
            feature_mode=feature_mode,
            target_kind="return",
            lookback=lookback,
            max_epochs=max_epochs,
            batch_size=batch_size
        )
        results.append({"feature_mode": feature_mode, "lookback": lookback, "mu_B_pred": pred})

results_df = pd.DataFrame(results)
print(f"Predicted next-day return for {asset} across configurations:")
display(results_df)

# 15. Step 5: Oracle ex-post (mu_true, Sigma_true)

Oracle benchmark uses **future realized returns** in `(t0 ... t0+m)`.

Important:
- This is not available at decision time.
- So it is not investable.
- We use it only to compare estimators ex-post.


In [ ]:
# 16. Compute mu_true and Sigma_true from future window

mu_true = future_window.mean()
Sigma_true = future_window.cov()

print("mu_true (oracle mean in future window):")
display(mu_true.to_frame(name="mu_true"))

print("Sigma_true (oracle covariance in future window):")
display(Sigma_true)


# 17. Step 6: Markowitz (Max Sharpe), long-only constraints

Portfolio weights:
- One weight per asset.
- Sum of weights must be 1.
- Long-only: each weight in [0, 1].

Sharpe ratio (simple daily version):
- `Sharpe = (portfolio_return - rf_daily) / portfolio_volatility`

We use `scipy.optimize.minimize` with SLSQP.


In [ ]:
# 18. Portfolio functions + constraints

def portfolio_return(w, mu):
    return float(np.dot(w, mu))

def portfolio_vol(w, Sigma):
    return float(np.sqrt(np.dot(w.T, np.dot(Sigma, w))))

def sharpe_ratio(w, mu, Sigma, rf_daily):
    vol = portfolio_vol(w, Sigma)
    if vol <= 1e-12:
        return -1e9
    ret = portfolio_return(w, mu)
    return (ret - rf_daily) / vol

def negative_sharpe(w, mu, Sigma, rf_daily):
    return -sharpe_ratio(w, mu, Sigma, rf_daily)

def solve_max_sharpe(mu, Sigma, rf_daily):
    n_assets = len(mu)
    w0 = np.ones(n_assets) / n_assets

    bounds = [(0.0, 1.0)] * n_assets
    constraints = ({"type": "eq", "fun": lambda w: np.sum(w) - 1.0},)

    result = minimize(
        negative_sharpe,
        w0,
        args=(mu.values, Sigma.values, rf_daily),
        method="SLSQP",
        bounds=bounds,
        constraints=constraints
    )

    if not result.success:
        raise RuntimeError(f"Optimization failed: {result.message}")

    return result.x


In [ ]:
# 19. Solve optimization for A, B and Oracle cases

rf_annual = 0.02
rf_daily = rf_annual / 252

w_A = solve_max_sharpe(mu_A, Sigma_A, rf_daily)
w_B = solve_max_sharpe(mu_B, Sigma_B, rf_daily)
w_T = solve_max_sharpe(mu_true, Sigma_true, rf_daily)

weights_table = pd.DataFrame({
    "Ticker": tickers,
    "w_A_baseline": w_A,
    "w_B_lstm": w_B,
    "w_T_oracle": w_T
})

print("Portfolio weights (Max Sharpe, long-only):")
display(weights_table)


In [ ]:
# 20. Exercise 4 (TODO)
# Check sum(weights)=1 for each portfolio and check all weights in [0, 1].

for name, w in [("w_A_baseline", w_A), ("w_B_lstm", w_B), ("w_T_oracle", w_T)]:
    sum_ok = np.isclose(np.sum(w), 1.0, atol=1e-6)
    bounds_ok = np.all((w >= -1e-6) & (w <= 1.0 + 1e-6))
    print(f"{name}: sum={np.sum(w):.8f} (ok={sum_ok}) | all in [0,1]: {bounds_ok}")

# 21. Step 7: Comparison versus oracle

Metrics:
- Weight distances: L1 and L2 between estimated portfolio and oracle portfolio.
- Sharpe regret under oracle parameters:
  - `regret_A = Sharpe(w_T; mu_true, Sigma_true) - Sharpe(w_A; mu_true, Sigma_true)`
  - `regret_B = Sharpe(w_T; mu_true, Sigma_true) - Sharpe(w_B; mu_true, Sigma_true)`
- Mean return prediction error:
  - `MAE(mu_A, mu_true)` and `MAE(mu_B, mu_true)`


In [ ]:
# 22. Compute comparison metrics

# Weight distances vs oracle
dA_L1 = np.sum(np.abs(w_A - w_T))
dA_L2 = np.sqrt(np.sum((w_A - w_T) ** 2))

dB_L1 = np.sum(np.abs(w_B - w_T))
dB_L2 = np.sqrt(np.sum((w_B - w_T) ** 2))

# Sharpe values under oracle parameters
S_T = sharpe_ratio(w_T, mu_true.values, Sigma_true.values, rf_daily)
S_A_true = sharpe_ratio(w_A, mu_true.values, Sigma_true.values, rf_daily)
S_B_true = sharpe_ratio(w_B, mu_true.values, Sigma_true.values, rf_daily)

regret_A = S_T - S_A_true
regret_B = S_T - S_B_true

# Mean absolute error in mu vectors
mae_mu_A = np.mean(np.abs(mu_A.values - mu_true.values))
mae_mu_B = np.mean(np.abs(mu_B.values - mu_true.values))

comparison_table = pd.DataFrame({
    "Case": ["A: Baseline", "B: LSTM"],
    "L1 distance to oracle weights": [dA_L1, dB_L1],
    "L2 distance to oracle weights": [dA_L2, dB_L2],
    "Sharpe regret (oracle params)": [regret_A, regret_B],
    "MAE(mu_hat, mu_true)": [mae_mu_A, mae_mu_B]
})

print("Sharpe with oracle parameters:")
print(f"S_T (oracle optimal): {S_T:.6f}")
print(f"S_A_true: {S_A_true:.6f}")
print(f"S_B_true: {S_B_true:.6f}")

print("\nFinal comparison table:")
display(comparison_table)


In [ ]:
# 23. Final visualization: weight bars (A vs B vs Oracle)

x = np.arange(len(tickers))
width = 0.25

plt.figure(figsize=(10, 5))
plt.bar(x - width, w_A, width=width, label="w_A Baseline")
plt.bar(x, w_B, width=width, label="w_B LSTM")
plt.bar(x + width, w_T, width=width, label="w_T Oracle")

plt.xticks(x, tickers)
plt.ylabel("Weight")
plt.title("Portfolio Weights Comparison")
plt.legend()
plt.tight_layout()
plt.show()


# 24. Mini-final exercise (TODO): change the criterion

Now repeat the experiment using **Minimum Variance** instead of Max Sharpe.

Checklist:
1. Define objective: minimize portfolio variance.
2. Compute `w_A_minvar`, `w_B_minvar`, `w_T_minvar`.
3. Compare distance to oracle min-variance weights.



In [ ]:
# 24. Mini-final exercise (TODO): change the criterion
# Now repeat the experiment using Minimum Variance instead of Max Sharpe.

# Step 1: Define the minimum variance solver
def solve_min_variance(Sigma):
    n_assets = len(Sigma)

    # Start from equal weights
    w0 = np.ones(n_assets) / n_assets

    # Long-only: each weight between 0 and 1
    bounds = [(0.0, 1.0)] * n_assets

    # Weights must sum to 1
    constraints = ({"type": "eq", "fun": lambda w: np.sum(w) - 1.0},)

    result = minimize(
        lambda w, S: np.dot(w.T, np.dot(S, w)),
        w0,
        args=(Sigma.values,),
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"ftol": 1e-12, "maxiter": 1000}
    )

    if not result.success:
        raise RuntimeError(f"Optimization failed: {result.message}")

    return result.x


# Step 2: Compute min-variance weights for each case
w_A_minvar = solve_min_variance(Sigma_A)
w_B_minvar = solve_min_variance(Sigma_B)
w_T_minvar = solve_min_variance(Sigma_true)

print("Min-variance weights:")
minvar_table = pd.DataFrame({
    "Ticker": tickers,
    "w_A_minvar": w_A_minvar,
    "w_B_minvar": w_B_minvar,
    "w_T_minvar": w_T_minvar
})
display(minvar_table)


# Step 3: Compare distance to oracle min-variance weights
dA_L1_minvar = np.sum(np.abs(w_A_minvar - w_T_minvar))
dA_L2_minvar = np.sqrt(np.sum((w_A_minvar - w_T_minvar) ** 2))

dB_L1_minvar = np.sum(np.abs(w_B_minvar - w_T_minvar))
dB_L2_minvar = np.sqrt(np.sum((w_B_minvar - w_T_minvar) ** 2))

print("\nDistance to oracle min-variance weights:")
minvar_comparison = pd.DataFrame({
    "Case": ["A: Baseline", "B: LSTM"],
    "L1 distance to oracle weights": [dA_L1_minvar, dB_L1_minvar],
    "L2 distance to oracle weights": [dA_L2_minvar, dB_L2_minvar],
})
display(minvar_comparison)


# Plot weights
x = np.arange(len(tickers))
width = 0.25

plt.figure(figsize=(10, 5))
plt.bar(x - width, w_A_minvar, width=width, label="w_A MinVar")
plt.bar(x,         w_B_minvar, width=width, label="w_B MinVar (LSTM)")
plt.bar(x + width, w_T_minvar, width=width, label="w_T MinVar (Oracle)")
plt.xticks(x, tickers)
plt.ylabel("Weight")
plt.title("Min-Variance Portfolio Weights Comparison")
plt.legend()
plt.tight_layout()
plt.show()

There are few things worth noting from the results:

* w_A (baseline) concentrates heavily in GLD and EEM, ignoring SPY and TLT entirely.
* w_B (LSTM) spreads across SPY and TLT, which makes sense since the LSTM predicted lower variance for bonds.
* w_T (oracle) puts most weight in EEM and GLD as these happened to be the lowest-variance assets in the actual future window.
* B: LSTM distances are much larger than A (L1: 1.89 vs 0.32), meaning the LSTM's predicted covariance structure led it quite far from the true minimum variance portfolio in this particular window.